# PYTHON ANALYSIS : the effect of a prevention program on wrist extension and forearm supination in leather workers.
Student : DUPLAIX JAHINNA


This notebook contains the Python part of the movement analysis project.
It is used to:
- I. Inspect the raw dataset
- II. Clean the data
- III. Create derived variables
- IV. Produce basic exploratory outputs

First, we need to import the necessary libraries.

In [1]:
# This command installs the Python libraries needed for the project.
# - pandas: used to read and manipulate tables
# - openpyxl: used to read Excel files
# - matplotlib: used to create simple figures in Python
%pip install pandas openpyxl matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# This line imports the pandas library.
# Pandas is used to read and manipulate tables such as Excel files.
import pandas as pd

# PART 1 : DATA CLEANING AND PREPARATION

## I. Inspect the raw dataset

Now we need to import the data and analyze its structure.

In [3]:
# This line stores the path of the Excel file.
# The path is relative, which means the project can work on another computer without changing the path, as long as the file is in the "data" folder.
file_path = "data/TMinstitute_prevention_base_2025.xlsx"

# This line opens the Excel file without reading a specific sheet yet.
# It allows us to inspect the names of the sheets first.
excel_file = pd.ExcelFile(file_path)

# This line prints the names of all sheets in the Excel file.
# We use it to identify the sheet that contains the data we want.
print(excel_file.sheet_names)

['range_of_motion']


In [15]:

# This line reads the sheet called "range_of_motion" from the Excel file.
# The result is stored in a table called df.
df = pd.read_excel(file_path, sheet_name="range_of_motion")

# This line prints the first 5 rows of the dataset.
# It helps us see what the table looks like.
print(df.head())

# This line prints the number of rows and columns.
# It helps us understand the size of the dataset.
print("Shape:", df.shape)

# This line prints the names of all columns.
# It helps us verify that the expected variables are present.
print("Columns:", df.columns.tolist())

# This line prints the data type of each column.
# It helps us see whether a column is text, numeric, etc.
print(df.dtypes)

# This line counts the values present in the timepoint column.
# It helps us check if time labels are clean or if they need harmonization.
print(df["timepoint"].value_counts(dropna=False))

# Count missing values in the two movement variables
print(df[["wrist_extension_deg", "forearm_supination_deg"]].isna().sum())

# Count missing values in the structural variables
print(df[["participant_id", "timepoint"]].isna().sum())

# Count exact duplicate rows
print("Number of exact duplicate rows:", df.duplicated().sum())

  participant_id timepoint  wrist_extension_deg forearm_supination_deg
0           M001        T0                 51.9                   82.1
1           M001        T1                 55.3                   85.4
2           M002        T0                 43.2                   70.5
3           M002        T1                 44.7                   74.2
4           M003        T0                 47.2                   62.6
Shape: (950, 4)
Columns: ['participant_id', 'timepoint', 'wrist_extension_deg', 'forearm_supination_deg']
participant_id                str
timepoint                     str
wrist_extension_deg       float64
forearm_supination_deg     object
dtype: object
timepoint
T0     475
T1     470
T0       1
t0       1
T 1      1
t1       1
T 0      1
Name: count, dtype: int64
wrist_extension_deg       4
forearm_supination_deg    4
dtype: int64
participant_id    0
timepoint         0
dtype: int64
Number of exact duplicate rows: 4


Our database contains a single excel sheet "range_of_motion" with 950 rows and 4 columns. So we have 4 variables:

- participant_id
- timepoint
- wrist_extension_deg
- forearm_supination_deg

We can see that : 
- the timepoint labels are not fully harmonized
- there are duplicates because we don't have the same number of participants in TO and T1
- forearm_supination_deg is not yet stored as a numeric variable
- there are 8 missing values

The next step is to clean the dataset.

## II. Clean the dataset

### II.1. First, We have to harmonize the participant labels and timepoint labels.

In [5]:
# This line removes extra spaces before and after each participant ID, it helps standardize participant labels.
df["participant_id"] = df["participant_id"].astype(str).str.strip()

#This line removes extra spaces before and after each timepoint label, it helps standardize timepoint labels.
df["timepoint"] = df["timepoint"].astype(str).str.strip()

#This line replaces inconsistent versions of T0 and T1 with standard labels, it makes the timepoint column clean and usable for analysis.
df["timepoint"] = df["timepoint"].replace({"T 0 ": "T0", "T 0": "T0","T 1 ": "T1", "T 1": "T1", "t0": "T0", "t1": "T1"})

#This line prints the updated counts for the timepoint column, it allows us to check whether the labels are now harmonized.
print(df["timepoint"].value_counts(dropna=False))

timepoint
T0    478
T1    472
Name: count, dtype: int64


### II.2. We will make forearm_supination_deg numeric.

In [6]:
#We convert the supination column to text first, to be able to clean it.
df["forearm_supination_deg"] = df["forearm_supination_deg"].astype(str)

# We remove the degree symbol if it exists
df["forearm_supination_deg"] = df["forearm_supination_deg"].str.replace("°", "", regex=False)

# We remove spaces before and after the values
df["forearm_supination_deg"] = df["forearm_supination_deg"].str.strip()

# We convert the column to numeric values, if a value cannot be converted, it becomes NaN
df["forearm_supination_deg"] = pd.to_numeric(df["forearm_supination_deg"], errors="coerce")

# Now, we check the variable types
print(df.dtypes)

participant_id                str
timepoint                     str
wrist_extension_deg       float64
forearm_supination_deg    float64
dtype: object


We have two numeric variables and two text variables.

### II.3. We need to deal with duplicates.

In [7]:
# We count exact duplicate rows
print("Number of exact duplicate rows:", df.duplicated().sum())

# We looking for the number of rows before removing duplicates
print("Rows before removing duplicates:", len(df))

# We remove exact duplicate rows
df = df.drop_duplicates()

#We looking for the number of rows after removing duplicates
print("Rows after removing duplicates:", len(df))


Number of exact duplicate rows: 4
Rows before removing duplicates: 950
Rows after removing duplicates: 946


At this stage, the duplicated rows have been removed and the dataset contains the expected number of rows.

In [8]:
# We check missing values
print(df[["wrist_extension_deg", "forearm_supination_deg"]].isna().sum())
print(df[["participant_id", "timepoint"]].isna().sum())
# We have missing values in the 2 movement variables, we display the rows that contain at least one missing value.
missing_rows = df[
    df["wrist_extension_deg"].isna() | df["forearm_supination_deg"].isna()]

print(missing_rows)

wrist_extension_deg       4
forearm_supination_deg    4
dtype: int64
participant_id    0
timepoint         0
dtype: int64
    participant_id timepoint  wrist_extension_deg  forearm_supination_deg
165           M083        T1                  NaN                    70.6
307           M154        T1                 63.4                     NaN
315           M158        T1                  NaN                    74.6
394           M198        T0                 40.8                     NaN
452           M227        T0                  NaN                    76.2
602           M302        T0                  NaN                    67.4
694           M348        T0                 66.6                     NaN
894           M448        T0                 46.9                     NaN


We have 8 missing values in the 2 mouvement variables, for the moment we don't delete them. We will do it into wide format of the dataset because we don't want to break the participants structure.

### II.4. Inspect incomplete rows

Rows with missing movement values are displayed after cleaning.

This step helps identify which participant-timepoint combinations are incomplete before reshaping the dataset into wide format.

In [16]:
# We display rows with at least one missing movement value
missing_rows = df[
    df["wrist_extension_deg"].isna() | df["forearm_supination_deg"].isna()]

print(missing_rows)

    participant_id timepoint  wrist_extension_deg forearm_supination_deg
165           M083        T1                  NaN                   70.6
307           M154        T1                 63.4                    NaN
315           M158        T1                  NaN                   74.6
394           M198        T0                 40.8                    NaN
452           M227        T0                  NaN                   76.2
602           M302        T0                  NaN                   67.4
694           M348        T0                 66.6                    NaN
894           M448        T0                 46.9                    NaN


### II.5. Check the longitudinal structure of the dataset

Before reshaping the dataset into wide format, the participant structure is checked.

This step verifies that:
- each participant has a valid identifier
- each participant has exactly two rows
- each participant has two distinct timepoints
- there are no duplicated participant-timepoint combinations

This ensures that the longitudinal structure is valid before computing changes between T0 and T1.

In [10]:
# We check the number of rows per participant
rows_per_participant = df.groupby("participant_id").size()

print("Distribution of the number of rows per participant:")
print(rows_per_participant.value_counts().sort_index())

# We display participants who do not have exactly 2 rows
problem_rows = rows_per_participant[rows_per_participant != 2]
print("Participants with a number of rows different from 2:")
print(problem_rows)

# We check the number of distinct timepoints per participant
timepoints_per_participant = df.groupby("participant_id")["timepoint"].nunique()

print("Distribution of the number of distinct timepoints per participant:")
print(timepoints_per_participant.value_counts().sort_index())

# We display participants who do not have exactly 2 distinct timepoints
problem_timepoints = timepoints_per_participant[timepoints_per_participant != 2]
print("Participants with a number of distinct timepoints different from 2:")
print(problem_timepoints)

# We check duplicated participant-timepoint pairs
duplicate_pairs = df.duplicated(subset=["participant_id", "timepoint"]).sum()
print("Number of duplicated participant-timepoint pairs:", duplicate_pairs)

Distribution of the number of rows per participant:
2    473
Name: count, dtype: int64
Participants with a number of rows different from 2:
Series([], dtype: int64)
Distribution of the number of distinct timepoints per participant:
timepoint
1      3
2    470
Name: count, dtype: int64
Participants with a number of distinct timepoints different from 2:
participant_id
M148    1
M212    1
M285    1
Name: timepoint, dtype: int64
Number of duplicated participant-timepoint pairs: 3


#### II.5.a. Inspect participants with an invalid longitudinal structure
3 participants do not have 2 distinct timepoints.
They have two rows, but both rows correspond to the same timepoint.

These cases are inspected before reshaping the dataset into wide format.

In [11]:
# We display the rows of participants with an invalid longitudinal structure
problem_participants = ["M148", "M212", "M285"]

print(df[df["participant_id"].isin(problem_participants)].sort_values(["participant_id", "timepoint"]))

    participant_id timepoint  wrist_extension_deg  forearm_supination_deg
294           M148        T0                 50.3                    56.9
295           M148        T0                 50.4                    55.8
422           M212        T1                 53.1                    64.5
423           M212        T1                 56.0                    65.6
568           M285        T0                 52.9                    72.5
569           M285        T0                 52.8                    72.8


Our three participants have two rows but only one distinct timepoint:
- M148
- M212
- M285

Because a valid change between T0 and T1 cannot be computed for these participants, they are excluded before reshaping the dataset into wide format.

In [12]:
# WE store the IDs of participants with an invalid longitudinal structure
invalid_participants = ["M148", "M212", "M285"]

# We remove these participants from the cleaned long-format dataset
df_valid = df[~df["participant_id"].isin(invalid_participants)].copy()

# We display the number of rows in the valid long-format dataset
print("Rows in valid long-format dataset:", len(df_valid))

# We check the number of distinct timepoints per participant again
check_timepoints = df_valid.groupby("participant_id")["timepoint"].nunique()
print(check_timepoints.value_counts().sort_index())

Rows in valid long-format dataset: 940
timepoint
2    470
Name: count, dtype: int64


### II.6. Reshape the valid dataset into wide format

Only participants with a valid longitudinal structure are kept before reshaping.
The valid long-format dataset is converted into wide format so that each participant has one row with:
- wrist extension at T0
- wrist extension at T1
- forearm supination at T0
- forearm supination at T1

In [13]:
# We convert the valid long-format dataset to wide format
df_wide = df_valid.pivot(
    index="participant_id",
    columns="timepoint",
    values=["wrist_extension_deg", "forearm_supination_deg"]
)

# Flatten the multi-level column names
df_wide.columns = [f"{var}_{tp}" for var, tp in df_wide.columns]

# WE turn participant_id back into a normal column
df_wide = df_wide.reset_index()

# WE display the first rows and the size of the wide dataset
print(df_wide.head())
print("Shape of wide dataset:", df_wide.shape)

  participant_id  wrist_extension_deg_T0  wrist_extension_deg_T1  \
0           M001                    51.9                    55.3   
1           M002                    43.2                    44.7   
2           M003                    47.2                    47.8   
3           M004                    46.0                    45.1   
4           M005                    56.4                    59.1   

   forearm_supination_deg_T0  forearm_supination_deg_T1  
0                       82.1                       85.4  
1                       70.5                       74.2  
2                       62.6                       64.3  
3                       72.9                       74.4  
4                       65.2                       69.1  
Shape of wide dataset: (470, 5)


Now we have our wide dataset with 470 rows (participants) and 5 columns (variables).

The next step is to inspect the rows that still contain missing values. It's to identifies which participants still have incomplete movement data.

### II.7. We looking again for missing values

In [ ]:
# We count missing values in the wide dataset
print(df_wide.isna().sum())

participant_id               0
wrist_extension_deg_T0       2
wrist_extension_deg_T1       2
forearm_supination_deg_T0    3
forearm_supination_deg_T1    1
dtype: int64


At this step, we have 8 missing values.
Because the statistical analyses require complete measurements at both timepoints for both variables, a final complete-case analysis dataset is created.

Only participants with complete values for:
- wrist extension at T0 and T1
- forearm supination at T0 and T1

are kept for the final analyses.



### II.8. Create the final analysis dataset

The statistical analyses require complete values for both movement variables at both timepoints.
A complete-case analysis dataset is therefore created by keeping only participants with complete data on all four movement variables.

In [17]:
# We keep only participants with complete data on all four movement variables
analysis_df = df_wide.dropna(
    subset=[
        "wrist_extension_deg_T0",
        "wrist_extension_deg_T1",
        "forearm_supination_deg_T0",
        "forearm_supination_deg_T1"]
).copy()

# Display the number of participants kept for analysis
print("Rows in analysis dataset:", len(analysis_df))

# Display the number of participants removed because of missing values
print("Rows removed because of missing values:", len(df_wide) - len(analysis_df))

Rows in analysis dataset: 462
Rows removed because of missing values: 8


### II.9. Cleaning summary

The data cleaning and preparation steps resulted in a valid dataset for the final analyses.

Starting from the raw long-format dataset:

- exact duplicate rows were removed
- timepoint labels were harmonized
- movement variables were converted to numeric format
- participants with an invalid longitudinal structure were excluded
- participants with incomplete movement data were excluded from the final complete-case analysis dataset

In [18]:
# Print a simple summary of the cleaning and selection steps
print("Initial number of participants:", 473)
print("Participants excluded because of invalid longitudinal structure:", 3)
print("Participants remaining after structural check:", 470)
print("Participants excluded because of missing movement data:", 8)
print("Final number of participants in analysis dataset:", len(analysis_df))

Initial number of participants: 473
Participants excluded because of invalid longitudinal structure: 3
Participants remaining after structural check: 470
Participants excluded because of missing movement data: 8
Final number of participants in analysis dataset: 462


The final analysis dataset contains 462 participants with complete values for:

- wrist extension at T0 and T1
- forearm supination at T0 and T1

This dataset is now ready for the computation of the primary outcome variables and the exploratory analyses.